# 🫁 Chest Disease Detection
**Model:** EfficientNet-B0 with Grad-CAM  
**Dataset:** NIH Chest X-Ray14  
**Task:** Multi-label classification (14 pathology classes)  

**Runtime:** GPU (T4 recommended)  

**Techniques:**
- Transfer learning from ImageNet
- Multi-label binary cross-entropy loss
- Grad-CAM: visualises which regions triggered each prediction
- AUC per pathology class

In [ ]:
!git clone https://github.com/Aurovindhya/ml-portfolio-suite.git
%cd ml-portfolio-suite
!pip install -q torch torchvision timm scikit-learn matplotlib seaborn pillow

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

In [ ]:
# NIH Chest X-Ray14 — download a small subset via kaggle or use synthetic
# Full dataset: https://www.kaggle.com/datasets/nih-chest-xrays/data
# !pip install kaggle
# !kaggle datasets download -d nih-chest-xrays/data --unzip -p data/chest_xray

# Synthetic demo: create fake grayscale 'X-rays' for structure demo
import os, numpy as np
from PIL import Image
import sys; sys.path.insert(0, '.')
from modules.classification.chest_disease.model import PATHOLOGY_CLASSES

print('Pathology classes:')
for i, cls in enumerate(PATHOLOGY_CLASSES):
    print(f'  {i:02d}: {cls}')

In [ ]:
# Model architecture
from modules.classification.chest_disease.model import ChestDiseaseModel

model = ChestDiseaseModel(pretrained=True)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total:,}')
print(f'Trainable params: {trainable:,}')
print(f'Backbone: EfficientNet-B0 (ImageNet)')
print(f'Head: Linear(1280→512) → ReLU → Dropout → Linear(512→{len(PATHOLOGY_CLASSES)})')
print(f'Loss: Binary cross-entropy (multi-label)')

In [ ]:
# Training loop skeleton
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from modules.classification.chest_disease.model import get_transforms, ChestDiseaseModel, PATHOLOGY_CLASSES

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = ChestDiseaseModel(pretrained=True).to(device)

# Multi-label loss
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=10)

print('Training config:')
print('  Loss:      BCEWithLogitsLoss (multi-label)')
print('  Optimizer: Adam (lr=1e-4)')
print('  Scheduler: CosineAnnealingLR')
print('  Metric:    AUC per class + macro AUC')
print()
print('To train on NIH Chest X-Ray14:')
print('  1. Download dataset from Kaggle (link above)')
print('  2. Build DataLoader with multi-hot label vectors')
print('  3. Run training loop below')
print('  4. Save weights → weights/chest_disease.pth')

In [ ]:
# Grad-CAM demo on a synthetic image
import torch
import numpy as np
import matplotlib.pyplot as plt
from modules.classification.chest_disease.model import ChestDiseaseModel, get_transforms, PATHOLOGY_CLASSES
from PIL import Image
import cv2

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = ChestDiseaseModel(pretrained=True).to(device)

# Create synthetic X-ray-like grayscale image
dummy = np.random.randint(50, 200, (224, 224), dtype=np.uint8)
dummy_rgb = np.stack([dummy]*3, axis=-1)
img = Image.fromarray(dummy_rgb)

tf = get_transforms(train=False)
tensor = tf(img).unsqueeze(0).to(device)

# Generate Grad-CAM for 'Pneumonia' class (index 6)
cam = model.grad_cam(tensor, class_idx=6)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(dummy, cmap='gray'); axes[0].set_title('Input X-ray'); axes[0].axis('off')
axes[1].imshow(cam, cmap='jet'); axes[1].set_title('Grad-CAM heatmap'); axes[1].axis('off')

# Overlay
cam_resized = cv2.resize(cam, (224, 224))
heatmap = plt.cm.jet(cam_resized)[:,:,:3]
overlay = 0.5 * dummy_rgb / 255.0 + 0.5 * heatmap
axes[2].imshow(overlay.clip(0,1)); axes[2].set_title('Grad-CAM overlay'); axes[2].axis('off')

plt.suptitle('Grad-CAM: regions driving "Pneumonia" prediction', fontsize=12)
plt.tight_layout(); plt.show()